# AEGEAN | TripAdvisor_scraper_apify

Retrieve Apify reviews, normalize records and rebuild the combined dataset.

**Dependencies:** apify-client pandas. Install in the active kernel before running.

**Execution:** Run the setup, configuration and collection cells in order. Review airline selections and limits before collection. Existing data must be migrated to `Data/Scraping/tripadvisor_reviews/` before resuming.

**Outputs:** All generated files are stored in `Data/Scraping/tripadvisor_reviews/`. Existing filenames and schemas are retained.

**Cost warning:** Collection may start paid Apify jobs. Configure `APIFY_TOKEN` or enter it securely when prompted.

## 1. Portable project paths

In [ ]:
from pathlib import Path

# Locate the project from the notebook kernel's working directory.
# Start Jupyter inside the project tree; the root name is not significant.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if all((p / name).is_dir() for name in ("Python Code", "Data", "Streamlit"))),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the project folder containing Python Code, Data and Streamlit.")

SCRAPING_DIR = PROJECT_ROOT / "Data" / "Scraping"
SOURCE_DIR = SCRAPING_DIR / 'tripadvisor_reviews'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)


## Imports and prerequisites

In [ ]:
import os
import re
import json
import getpass
import time
import traceback
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from apify_client import ApifyClient

## 1. Configuration

In [ ]:
ACTOR_ID = "zenomastro/tripadvisor-reviews-reliable"

# Enter your Apify token securely
APIFY_TOKEN = os.getenv("APIFY_TOKEN")

if not APIFY_TOKEN:
    APIFY_TOKEN = getpass.getpass("Enter Apify API token: ")

client = ApifyClient(APIFY_TOKEN)


# Tripadvisor airline URLs
#
# IMPORTANT:
# The AEGEAN URL has already been tested successfully.
# Verify the other three URLs before running a paid job.
#
# Replace None with the verified Tripadvisor Airline Review URL.

AIRLINE_URLS = {
    "Aegean Airlines":
        "https://www.tripadvisor.com/Airline_Review-d8728985-Reviews-AEGEAN",

    "Air France":
        "https://www.tripadvisor.com/Airline_Review-d8729003-Reviews-Air-France",

    "Lufthansa":
        "https://www.tripadvisor.com/Airline_Review-d8729113-Reviews-Lufthansa",

    "Turkish Airlines":
        "https://www.tripadvisor.com/Airline_Review-d8729174-Reviews-Turkish-Airlines",

    "Bangkok Airways":
        "https://www.tripadvisor.com/Airline_Review-d8729031-Reviews-Bangkok-Airways",
    
    "Porter":
        "https://www.tripadvisor.com/Airline_Review-d8728944-Reviews-Porter-Airlines",
    
    "Virgin":
        "https://www.tripadvisor.com/Airline_Review-d8728931-Reviews-Virgin-Australia"
    
}


# ------------------------------------------------------------
# SELECT AIRLINES
# ------------------------------------------------------------

# Options:
# ["Air France"]
# ["Lufthansa", "Turkish Airlines"]
# ["Aegean Airlines", "Air France", "Lufthansa", "Turkish Airlines"]

AIRLINES_TO_SCRAPE = [
#    "Aegean Airlines",
#    "Air France",
#    "Lufthansa",
#    "Turkish Airlines",
    "Bangkok Airways",
    "Porter",
    "Virgin"
]


# ------------------------------------------------------------
# DATE FILTERS
# ------------------------------------------------------------

START_DATE = "2024-01-01"

# None means no upper date filter
END_DATE = None


# ------------------------------------------------------------
# REVIEW LIMITS
# ------------------------------------------------------------

MAX_REVIEWS_PER_AIRLINE = 10000
MAX_PAGES_PER_AIRLINE = 500

# Set to False to inspect the configuration without
# starting paid Apify runs.
EXECUTE_SCRAPING = True


# ------------------------------------------------------------
# OUTPUT SETTINGS
# ------------------------------------------------------------

OUTPUT_DIR = SOURCE_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_FILE = OUTPUT_DIR / "tripadvisor_all_airlines.csv"
SUMMARY_FILE = OUTPUT_DIR / "tripadvisor_scraping_summary.csv"
RUN_HISTORY_FILE = OUTPUT_DIR / "tripadvisor_run_history.csv"

# Your existing AEGEAN dataset
EXISTING_AEGEAN_FILE = Path(
    SOURCE_DIR / "tripadvisor_aegean_reviews.csv"
)


# ------------------------------------------------------------
# OPTIONAL: RESUME EXISTING APIFY RUNS
# ------------------------------------------------------------

# If a paid Apify run has already completed, add its run ID.
# The script will retrieve its dataset rather than starting
# a new paid run for that airline.
#
# Example:
# EXISTING_RUN_IDS = {
#     "Air France": "your_completed_run_id"
# }

EXISTING_RUN_IDS = {}


print("Configuration loaded.")
print("Airlines selected:", AIRLINES_TO_SCRAPE)
print("Observation period:", START_DATE, "to", END_DATE or "latest")
print("Maximum reviews per airline:", MAX_REVIEWS_PER_AIRLINE)
print("Paid scraping enabled:", EXECUTE_SCRAPING)

## 2. Helper Functions

In [ ]:
def airline_slug(name):
    """Create a filesystem-friendly airline name."""
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        name.lower()
    ).strip("_")


def airline_output_path(airline):
    """Return the output CSV path for the selected airline."""
    return OUTPUT_DIR / f"{airline_slug(airline)}_reviews.csv"


def normalize_dates(series):
    """
    Handle strings, timestamps and mixed date formats.
    Prevents the str vs Timestamp comparison error.
    """
    return pd.to_datetime(
        series,
        errors="coerce",
        format="mixed"
    )


def normalize_review_ids(series):
    """
    Standardize review IDs as strings.

    Avoids duplicate mismatches caused by reading IDs
    as integers, floats or strings.
    """
    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def read_existing_csv(path):
    """Read an existing CSV with consistent data types."""

    if not path.exists():
        return pd.DataFrame()

    df = pd.read_csv(
        path,
        dtype={"source_review_id": "string"},
        low_memory=False
    )

    if "review_date" in df.columns:
        df["review_date"] = normalize_dates(
            df["review_date"]
        )

    if "source_review_id" in df.columns:
        df["source_review_id"] = normalize_review_ids(
            df["source_review_id"]
        )

    return df


def merge_reviews(existing_df, new_df):
    """
    Merge datasets and remove duplicate reviews.

    Deduplication key:
        airline + source_review_id
    """

    if existing_df.empty:
        combined = new_df.copy()

    elif new_df.empty:
        combined = existing_df.copy()

    else:
        combined = pd.concat(
            [existing_df, new_df],
            ignore_index=True
        )

    if combined.empty:
        return combined

    combined["review_date"] = normalize_dates(
        combined["review_date"]
    )

    combined["source_review_id"] = normalize_review_ids(
        combined["source_review_id"]
    )

    combined = combined.drop_duplicates(
        subset=["airline", "source_review_id"],
        keep="first"
    )

    combined = combined.sort_values(
        ["airline", "review_date"],
        ascending=[True, False],
        na_position="last"
    ).reset_index(drop=True)

    return combined


def append_csv_records(path, new_rows, dedup_columns):
    """
    Append metadata/history records without duplicates.
    """

    new_df = pd.DataFrame(new_rows)

    if new_df.empty:
        return

    if path.exists():
        old_df = pd.read_csv(path)
        new_df = pd.concat(
            [old_df, new_df],
            ignore_index=True
        )

    new_df = new_df.drop_duplicates(
        subset=dedup_columns,
        keep="last"
    )

    new_df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

## 3. Standardize Tripadvisor Reviews

In [ ]:
RENAME_MAP = {
    "reviewId": "source_review_id",
    "reviewUrl": "review_url",
    "title": "review_title",
    "text": "review",
    "rating": "rating",
    "publishedDate": "review_date",
    "stayDate": "travel_date",
    "tripType": "trip_type",
    "language": "language",
    "originalLanguage": "original_language",
    "reviewerName": "author",

    "subratings.Legroom": "legroom",
    "subratings.Seat comfort": "seat_comfort",
    "subratings.Customer service": "customer_service",
    "subratings.Value for money": "value_for_money",
    "subratings.Cleanliness": "cleanliness",
    "subratings.Check-in and boarding": "check_in_boarding",
    "subratings.Food and Beverage": "food_beverages",
    "subratings.In-flight Entertainment": "entertainment"
}


def standardize_reviews(items, airline, run_id):

    """Transform source records into the existing normalized output schema."""
    raw = pd.json_normalize(items)

    if raw.empty:
        return pd.DataFrame()

    if "recordType" not in raw.columns:
        raise ValueError(
            "Actor dataset does not contain recordType."
        )

    # Exclude place_summary and review_insights
    reviews = raw[
        raw["recordType"] == "review"
    ].copy()

    if reviews.empty:
        return pd.DataFrame()

    # Rename fields
    reviews = reviews.rename(
        columns=RENAME_MAP
    )

    # Ensure essential fields exist
    required_columns = [
        "source_review_id",
        "review_date",
        "rating",
        "review",
        "review_title"
    ]

    for col in required_columns:
        if col not in reviews.columns:
            reviews[col] = pd.NA

    # Standardize types
    reviews["source_review_id"] = normalize_review_ids(
        reviews["source_review_id"]
    )

    reviews["review_date"] = normalize_dates(
        reviews["review_date"]
    )

    reviews["rating"] = pd.to_numeric(
        reviews["rating"],
        errors="coerce"
    )

    # Add metadata
    reviews["airline"] = airline
    reviews["source"] = "Tripadvisor"
    reviews["apify_run_id"] = run_id
    reviews["scraped_at"] = datetime.now(
        timezone.utc
    ).isoformat()

    # Remove records without valid review IDs
    reviews = reviews[
        reviews["source_review_id"].notna()
        & reviews["source_review_id"].ne("")
    ].copy()

    # Filter observation period
    reviews = reviews[
        reviews["review_date"] >= pd.Timestamp(START_DATE)
    ].copy()

    if END_DATE is not None:
        reviews = reviews[
            reviews["review_date"] <= pd.Timestamp(END_DATE)
        ].copy()

    # Remove duplicates within current run
    reviews = reviews.drop_duplicates(
        subset=["airline", "source_review_id"],
        keep="first"
    )

    # Put main columns first
    main_columns = [
        "airline",
        "source_review_id",
        "review_date",
        "rating",
        "review_title",
        "review",
        "source",
        "review_url",
        "travel_date",
        "trip_type",
        "language",
        "original_language",
        "author",
        "apify_run_id",
        "scraped_at"
    ]

    existing_main = [
        c for c in main_columns
        if c in reviews.columns
    ]

    remaining = [
        c for c in reviews.columns
        if c not in existing_main
    ]

    reviews = reviews[
        existing_main + remaining
    ]

    return reviews

## 4. Actor Running / Retrieval

In [ ]:
def get_apify_reviews(airline, url):

    """Retrieve records from a new or previously completed Apify run."""
    print("\n" + "=" * 75)
    print("AIRLINE:", airline)
    print("=" * 75)

    # Reuse an existing run if configured
    if airline in EXISTING_RUN_IDS:

        run_id = EXISTING_RUN_IDS[airline]

        print("Reusing existing Apify run:", run_id)

        run = client.run(run_id).get()

    else:

        if not EXECUTE_SCRAPING:
            print("DRY RUN: No paid extraction started.")
            return None, []

        run_input = {
            "placeUrls": [url],
            "maxReviewsPerPlace": MAX_REVIEWS_PER_AIRLINE,
            "maxTotalReviews": MAX_REVIEWS_PER_AIRLINE,
            "maxPagesPerPlace": MAX_PAGES_PER_AIRLINE,
            "reviewedAfter": START_DATE,
            "languageCodes": [],
            "machineTranslation": False,
            "includePlaceSummary": True,
            "autoProxyFallback": True
        }

        if END_DATE is not None:
            run_input["reviewedBefore"] = END_DATE

        print("Starting Apify actor...")
        print("Maximum reviews:", MAX_REVIEWS_PER_AIRLINE)

        run = client.actor(ACTOR_ID).call(
            run_input=run_input
        )

        run_id = run.id

    print("Run status:", run.status)
    print("Run ID:", run_id)

    # Save run metadata immediately
    append_csv_records(
        RUN_HISTORY_FILE,
        [{
            "airline": airline,
            "run_id": run_id,
            "status": str(run.status),
            "retrieved_at": datetime.now(
                timezone.utc
            ).isoformat()
        }],
        dedup_columns=["run_id"]
    )

    if str(run.status).upper() != "SUCCEEDED":
        print(
            "Warning: Run did not finish successfully. "
            "Attempting to retrieve any available records."
        )

    dataset_id = run.default_dataset_id

    if not dataset_id:
        raise ValueError(
            f"No dataset ID available for run {run_id}"
        )

    items = list(
        client.dataset(dataset_id).iterate_items()
    )

    print("Total dataset records:", len(items))

    return run_id, items

## 5. Data Quality / Coverage Summary

In [ ]:
def create_airline_summary(airline, df):

    """Compute review coverage statistics for an airline."""
    if df.empty:
        return {
            "airline": airline,
            "total_reviews": 0,
            "earliest_date": None,
            "latest_date": None,
            "reviews_2024": 0,
            "reviews_2025": 0,
            "reviews_2026": 0,
            "missing_text": 0,
            "missing_rating": 0,
            "start_date_reached": False
        }

    dates = normalize_dates(df["review_date"])

    yearly_counts = dates.dt.year.value_counts()

    earliest = dates.min()
    latest = dates.max()

    # Reaching Jan 1 is a useful diagnostic, but it
    # does not prove all reviews were collected.
    start_reached = (
        pd.notna(earliest)
        and earliest.normalize() <= pd.Timestamp(START_DATE)
    )

    return {
        "airline": airline,
        "total_reviews": len(df),
        "earliest_date": earliest,
        "latest_date": latest,
        "reviews_2024": int(yearly_counts.get(2024, 0)),
        "reviews_2025": int(yearly_counts.get(2025, 0)),
        "reviews_2026": int(yearly_counts.get(2026, 0)),
        "missing_text": int(
            df["review"].isna().sum()
            + df["review"].fillna("").astype(str).str.strip().eq("").sum()
            - df["review"].isna().sum()
        ),
        "missing_rating": int(
            df["rating"].isna().sum()
        ),
        "start_date_reached": bool(start_reached)
    }


def display_airline_summary(summary):

    """Display the calculated coverage statistics."""
    print("\n" + "-" * 60)
    print("COLLECTION SUMMARY")
    print("-" * 60)

    for key, value in summary.items():
        print(f"{key}: {value}")

    if summary["start_date_reached"]:
        print(
            "\nDate boundary reached. "
            "Completeness is not independently verified."
        )
    else:
        print(
            "\nDate boundary not reached. "
            "Additional historical reviews may be needed."
        )

## 6. Import Your Existing Aegean Dataset

In [ ]:
def import_existing_aegean():

    """Merge the legacy AEGEAN export into its standard output file."""
    if not EXISTING_AEGEAN_FILE.exists():
        print("No existing AEGEAN CSV found.")
        return

    print("\nImporting existing AEGEAN reviews...")

    aegean = read_existing_csv(
        EXISTING_AEGEAN_FILE
    )

    if aegean.empty:
        print("Existing AEGEAN CSV is empty.")
        return

    aegean["airline"] = "Aegean Airlines"
    aegean["source"] = "Tripadvisor"

    aegean_path = airline_output_path(
        "Aegean Airlines"
    )

    old = read_existing_csv(aegean_path)

    merged = merge_reviews(old, aegean)

    merged.to_csv(
        aegean_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "AEGEAN unique reviews imported:",
        len(merged)
    )

## 7. Main Multi-Airline Scraper

In [ ]:
def run_multi_airline_scraper():

    # Preserve the existing 1,521 AEGEAN reviews
    """Run the configured airline collection jobs and record their outcomes."""
    import_existing_aegean()

    results = []

    for index, airline in enumerate(
        AIRLINES_TO_SCRAPE,
        start=1
    ):

        print("\n" + "#" * 75)
        print(
            f"Processing airline {index}/"
            f"{len(AIRLINES_TO_SCRAPE)}: {airline}"
        )
        print("#" * 75)

        if airline not in AIRLINE_URLS:
            print("Unknown airline:", airline)
            continue

        url = AIRLINE_URLS[airline]

        if not url:
            print(
                "SKIPPED: No verified Tripadvisor URL "
                "configured for this airline."
            )
            continue

        try:

            # Step 1: Run or reuse Apify extraction
            run_id, items = get_apify_reviews(
                airline,
                url
            )

            if run_id is None:
                continue

            # Step 2: Standardize new reviews
            new_reviews = standardize_reviews(
                items,
                airline,
                run_id
            )

            print(
                "New unique reviews returned:",
                len(new_reviews)
            )

            # Step 3: Load existing airline data
            output_file = airline_output_path(
                airline
            )

            existing_reviews = read_existing_csv(
                output_file
            )

            # Step 4: Merge and deduplicate
            combined_reviews = merge_reviews(
                existing_reviews,
                new_reviews
            )

            # Step 5: Save immediately
            combined_reviews.to_csv(
                output_file,
                index=False,
                encoding="utf-8-sig"
            )

            print("Saved:", output_file.resolve())

            # Step 6: Report date coverage
            summary = create_airline_summary(
                airline,
                combined_reviews
            )

            display_airline_summary(summary)

            results.append(summary)

        except Exception as e:

            print(
                f"\nERROR processing {airline}: {e}"
            )

            traceback.print_exc()

            # Continue to next airline
            continue

    return pd.DataFrame(results)

## 8. Build Combined Dataset

In [ ]:
def build_combined_dataset():

    """Rebuild the combined dataset from saved airline CSVs."""
    frames = []

    for airline in AIRLINE_URLS:

        path = airline_output_path(airline)

        if not path.exists():
            continue

        df_airline = read_existing_csv(path)

        if df_airline.empty:
            continue

        df_airline["airline"] = airline

        frames.append(df_airline)

    if not frames:
        print("No airline datasets found.")
        return pd.DataFrame()

    df_all = pd.concat(
        frames,
        ignore_index=True
    )

    df_all = merge_reviews(
        pd.DataFrame(),
        df_all
    )

    df_all.to_csv(
        COMBINED_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n" + "=" * 75)
    print("FINAL COMBINED DATASET")
    print("=" * 75)

    print("Total reviews:", len(df_all))

    print("\nReviews by airline:")
    print(
        df_all["airline"].value_counts()
    )

    print("\nReviews by year and airline:")
    print(
        pd.crosstab(
            df_all["airline"],
            df_all["review_date"].dt.year
        )
    )

    print("\nSaved combined CSV:")
    print(COMBINED_FILE.resolve())

    return df_all

## 9. Execute

In [ ]:
scraping_summary = run_multi_airline_scraper()

df_tripadvisor_all = build_combined_dataset()

if not df_tripadvisor_all.empty:

    # Generate coverage summary for every saved airline
    all_summaries = []

    for airline, group in df_tripadvisor_all.groupby(
        "airline"
    ):
        all_summaries.append(
            create_airline_summary(airline, group)
        )

    df_summary = pd.DataFrame(all_summaries)

    df_summary.to_csv(
        SUMMARY_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nFINAL COVERAGE SUMMARY:")
    display(df_summary)

    print("\nSample reviews:")
    display(
        df_tripadvisor_all[
            [
                "airline",
                "review_date",
                "rating",
                "review_title",
                "review"
            ]
        ].head(10)
    )